In [1]:
# Step 1: Import the Pandas library (our data spreadsheet engine)
import pandas as pd

# Step 2: Define our core business question as a comment for documentation
# "Identify transactions in Q1–Q2 2025 that show patterns consistent with structuring, account takeover, or unusual customer behaviour."

# Step 3: Load the dataset
# Because our notebook is in the 'notebooks' folder and the data is in the 'data' folder, we use '../data/' to look one folder up.
df = pd.read_csv("../data/project1_banking_transactions.csv")

# Step 4: Display the first 5 rows to ensure it loaded correctly
df.head()

,transaction_id,transaction_date,transaction_time,customer_id,account_number,transaction_type,channel,branch,amount_ngn,balance_after_ngn,is_flagged_anomaly
0,TXN0001382,2025-06-20,01:59:29,CUST00279,3513198102,Bill Payment,ATM,Pantami,6097.08,-45127.94,0
1,TXN0001738,2025-04-16,17:19:59,CUST00123,4645008999,Transfer,USSD,Kano Central,48011.02,-50000.00,0
2,TXN0010487,2025-03-04,21:37:55,CUST01215,6461346263,POS Payment,Branch,Pantami,28823.85,117029.06,0
3,TXN0007358,2025-04-20,13:25:43,CUST00035,4929721883,Withdrawal,Mobile App,Gombe GRA,61720.96,175556.72,0
4,TXN0010066,2025-03-20,13:46:25,CUST00727,2797013429,Transfer,Mobile App,Tudun Wada,106151.91,252656.65,0


In [2]:
# Check the shape of the data (Rows, Columns)
print(f"Dataset Shape: {df.shape}\n")

# Check for any missing values across columns
print("Missing Values per Column:")
print(df.isnull().sum())
print("-" * 50)

# Save the ground-truth answers safely in a separate variable, then drop it from our working dataframe
ground_truth = df[['transaction_id', 'is_flagged_anomaly']].copy()
df = df.drop(columns=['is_flagged_anomaly'])

print("Successfully sequestered the 'is_flagged_anomaly' column for the final blind test!")

Dataset Shape: (10605, 11)

Missing Values per Column:
transaction_id        0
transaction_date      0
transaction_time      0
customer_id           0
account_number        0
transaction_type      0
channel               0
branch                0
amount_ngn            0
balance_after_ngn     0
is_flagged_anomaly    0
dtype: int64
--------------------------------------------------
Successfully sequestered the 'is_flagged_anomaly' column for the final blind test!


In [3]:
# Step 1: Combine the text of transaction_date and transaction_time with a space in between,
# and convert it into a true Python Datetime data type.
df['transaction_datetime'] = pd.to_datetime(df['transaction_date'] + ' ' + df['transaction_time'])

# Step 2: Drop the original separate text columns since they are now redundant
df = df.drop(columns=['transaction_date', 'transaction_time'])

# Step 3: Run df.info() to confirm our columns and see their exact structural data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10605 entries, 0 to 10604
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   transaction_id        10605 non-null  object        
 1   customer_id           10605 non-null  object        
 2   account_number        10605 non-null  int64         
 3   transaction_type      10605 non-null  object        
 4   channel               10605 non-null  object        
 5   branch                10605 non-null  object        
 6   amount_ngn            10605 non-null  float64       
 7   balance_after_ngn     10605 non-null  float64       
 8   transaction_datetime  10605 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 745.8+ KB


In [4]:
# Save our prepared dataframe as a new CSV file in our data folder
df.to_csv("../data/project1_banking_transactions_prepared.csv", index=False)

print("File successfully saved as 'project1_banking_transactions_prepared.csv'!")

File successfully saved as 'project1_banking_transactions_prepared.csv'!


In [5]:
# 1. Calculate the mean (average) and standard deviation for each transaction type
stats = df.groupby('transaction_type')['amount_ngn'].agg(['mean', 'std']).reset_index()

# 2. Merge these calculated limits back into our main dataframe
df_with_stats = df.merge(stats, on='transaction_type', how='left')

# 3. Create a clean flag: 1 if the amount exceeds 3 standard deviations above the mean, else 0
df_with_stats['flag_statistical_outlier'] = (
    df_with_stats['amount_ngn'] > (df_with_stats['mean'] + (3 * df_with_stats['std']))
).astype(int)

# 4. Count how many anomalies this pattern caught
statistical_outlier_count = df_with_stats['flag_statistical_outlier'].sum()
print(f"Pattern 1 Complete: Detected {statistical_outlier_count} statistical outliers.")

Pattern 1 Complete: Detected 48 statistical outliers.


In [7]:
# 1. Extract the specific hour (0 to 23) from our formatted datetime column
df_with_stats['hour'] = df_with_stats['transaction_datetime'].dt.hour

# 2. Define the rule: Anomaly if it's a Withdrawal AND the hour is between 0 (midnight) and 4 AM
df_with_stats['flag_off_hours_withdrawal'] = (
    (df_with_stats['transaction_type'] == 'Withdrawal') & 
    (df_with_stats['hour'] >= 0) & 
    (df_with_stats['hour'] < 4)
).astype(int)

# 3. Count how many anomalies this pattern caught
off_hours_count = df_with_stats['flag_off_hours_withdrawal'].sum()
print(f"Pattern 2 Complete: Detected {off_hours_count} off-hours withdrawal anomalies.")

Pattern 2 Complete: Detected 522 off-hours withdrawal anomalies.


In [7]:
# 1. Ensure our entire dataframe is strictly sorted chronologically per account
df_with_stats = df_with_stats.sort_values(by=['account_number', 'transaction_datetime']).reset_index(drop=True)

# 2. Isolate transfers into a temporary dataframe to calculate velocity accurately
transfers = df_with_stats[df_with_stats['transaction_type'] == 'Transfer'].copy()

# 3. For every account, pull the timestamp of the transaction that happened 2 steps prior
transfers['time_two_transfers_ago'] = transfers.groupby('account_number')['transaction_datetime'].shift(2)

# 4. Calculate the time difference in minutes between the current transfer and 2 transfers ago
transfers['velocity_minutes'] = (transfers['transaction_datetime'] - transfers['time_two_transfers_ago']).dt.total_seconds() / 60.0

# 5. Flag if 3 transfers occurred within a window of 10 minutes or less
transfers['flag_velocity_transfer'] = ((transfers['velocity_minutes'] <= 10) & (transfers['velocity_minutes'] >= 0)).astype(int)

# 6. Merge this specific flag back into our master dataframe and handle non-transfer rows with 0
df_with_stats = df_with_stats.merge(transfers[['transaction_id', 'flag_velocity_transfer']], on='transaction_id', how='left')
df_with_stats['flag_velocity_transfer'] = df_with_stats['flag_velocity_transfer'].fillna(0).astype(int)

velocity_count = df_with_stats['flag_velocity_transfer'].sum()
print(f"Pattern 3 Complete: Detected {velocity_count} velocity transfer anomalies.")

Pattern 3 Complete: Detected 44 velocity transfer anomalies.


In [5]:
import pandas as pd

print("Initializing Anomaly Detection Engine...")

df = pd.read_csv("../data/project1_banking_transactions.csv")

df['transaction_datetime'] = pd.to_datetime(df['transaction_date'] + ' ' + df['transaction_time'])
df = df.drop(columns=['transaction_date', 'transaction_time'])

ground_truth = df[['transaction_id', 'is_flagged_anomaly']].copy()
df = df.drop(columns=['is_flagged_anomaly'])

print(f"-> Successfully loaded {df.shape[0]} transactions. Answer key safely sequestered.")

stats = df.groupby('transaction_type')['amount_ngn'].agg(['mean', 'std']).reset_index()
df_with_stats = df.merge(stats, on='transaction_type', how='left')
df_with_stats['flag_statistical_outlier'] = (
    df_with_stats['amount_ngn'] > (df_with_stats['mean'] + (3 * df_with_stats['std']))
).astype(int)

df_with_stats['hour'] = df_with_stats['transaction_datetime'].dt.hour
df_with_stats['flag_off_hours_withdrawal'] = (
    (df_with_stats['transaction_type'] == 'Withdrawal') & 
    (df_with_stats['hour'] >= 0) & 
    (df_with_stats['hour'] < 4)
).astype(int)

df_with_stats = df_with_stats.sort_values(by=['account_number', 'transaction_datetime']).reset_index(drop=True)
transfers = df_with_stats[df_with_stats['transaction_type'] == 'Transfer'].copy()
transfers['time_two_transfers_ago'] = transfers.groupby('account_number')['transaction_datetime'].shift(2)
transfers['velocity_minutes'] = (transfers['transaction_datetime'] - transfers['time_two_transfers_ago']).dt.total_seconds() / 60.0
transfers['flag_velocity_transfer'] = ((transfers['velocity_minutes'] <= 10) & (transfers['velocity_minutes'] >= 0)).astype(int)

df_with_stats = df_with_stats.merge(transfers[['transaction_id', 'flag_velocity_transfer']], on='transaction_id', how='left')
df_with_stats['flag_velocity_transfer'] = df_with_stats['flag_velocity_transfer'].fillna(0).astype(int)

df_with_stats['transaction_date_only'] = df_with_stats['transaction_datetime'].dt.date
deposits = df_with_stats[df_with_stats['transaction_type'] == 'Deposit'][['transaction_id', 'account_number', 'transaction_date_only', 'amount_ngn']].rename(columns={'amount_ngn': 'dep_amt', 'transaction_id': 'dep_id'})
withdrawals = df_with_stats[df_with_stats['transaction_type'] == 'Withdrawal'][['transaction_id', 'account_number', 'transaction_date_only', 'amount_ngn']].rename(columns={'amount_ngn': 'with_amt', 'transaction_id': 'with_id'})

paired = deposits.merge(withdrawals, on=['account_number', 'transaction_date_only'], how='inner')
paired['is_round_trip'] = (abs(paired['dep_amt'] - paired['with_amt']) / paired['dep_amt'] <= 0.02)
true_round_trips = paired[paired['is_round_trip'] == True]
round_trip_ids = set(true_round_trips['dep_id']).union(set(true_round_trips['with_id']))
df_with_stats['flag_round_tripping'] = df_with_stats['transaction_id'].isin(round_trip_ids).astype(int)

df_with_stats['final_predicted_anomaly'] = (
    (df_with_stats['flag_statistical_outlier'] == 1) |
    (df_with_stats['flag_off_hours_withdrawal'] == 1) |
    (df_with_stats['flag_velocity_transfer'] == 1) |
    (df_with_stats['flag_round_tripping'] == 1)
).astype(int)

print("-> Logic filters executed. Anomaly aggregation complete.")

final_evaluation = df_with_stats.merge(ground_truth, on='transaction_id', how='left')

tp = ((final_evaluation['final_predicted_anomaly'] == 1) & (final_evaluation['is_flagged_anomaly'] == 1)).sum()
fp = ((final_evaluation['final_predicted_anomaly'] == 1) & (final_evaluation['is_flagged_anomaly'] == 0)).sum()
fn = ((final_evaluation['final_predicted_anomaly'] == 0) & (final_evaluation['is_flagged_anomaly'] == 1)).sum()

precision = (tp / (tp + fp)) * 100 if (tp + fp) > 0 else 0
recall = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0

print("\n" + "="*50)
print("             METRIC EVALUATION REPORT             ")
print("="*50)
print(f"True Fraud Instances Captured (True Positives):  {tp}")
print(f"False Alerts Raised (False Positives):           {fp}")
print(f"Missed Fraud Occurrences (False Negatives):      {fn}")
print("-"*50)
print(f"FINAL AML SYSTEM RECALL:")

Initializing Anomaly Detection Engine...
-> Successfully loaded 10605 transactions. Answer key safely sequestered.
-> Logic filters executed. Anomaly aggregation complete.

             METRIC EVALUATION REPORT             
True Fraud Instances Captured (True Positives):  116
False Alerts Raised (False Positives):           484
Missed Fraud Occurrences (False Negatives):      94
--------------------------------------------------
FINAL AML SYSTEM RECALL:


In [8]:
# 1. Create a clean date-only column to easily match same-day activity
df_with_stats['transaction_date_only'] = df_with_stats['transaction_datetime'].dt.date

# 2. Segment deposits and withdrawals into separate tracking tables
deposits = df_with_stats[df_with_stats['transaction_type'] == 'Deposit'][['transaction_id', 'account_number', 'transaction_date_only', 'amount_ngn']].rename(columns={'amount_ngn': 'dep_amt', 'transaction_id': 'dep_id'})
withdrawals = df_with_stats[df_with_stats['transaction_type'] == 'Withdrawal'][['transaction_id', 'account_number', 'transaction_date_only', 'amount_ngn']].rename(columns={'amount_ngn': 'with_amt', 'transaction_id': 'with_id'})

# 3. Perform an inner join on account number and date to pair up same-day matches
paired = deposits.merge(withdrawals, on=['account_number', 'transaction_date_only'], how='inner')

# 4. Apply our business rule: check if the withdrawal amount is within ±2% of the deposit amount
paired['is_round_trip'] = (abs(paired['dep_amt'] - paired['with_amt']) / paired['dep_amt'] <= 0.02)

# 5. Extract the transaction IDs involved in confirmed round-trips
true_round_trips = paired[paired['is_round_trip'] == True]
round_trip_ids = set(true_round_trips['dep_id']).union(set(true_round_trips['with_id']))

# 6. Apply the master flag to our main dataframe if a transaction matches those IDs
df_with_stats['flag_round_tripping'] = df_with_stats['transaction_id'].isin(round_trip_ids).astype(int)

round_trip_count = df_with_stats['flag_round_tripping'].sum()
print(f"Pattern 4 Complete: Detected {round_trip_count} round-tripping transactions.")

Pattern 4 Complete: Detected 0 round-tripping transactions.


In [9]:
# If any of our four custom feature flags equal 1, our final pipeline prediction is 1
df_with_stats['final_predicted_anomaly'] = (
    (df_with_stats['flag_statistical_outlier'] == 1) |
    (df_with_stats['flag_off_hours_withdrawal'] == 1) |
    (df_with_stats['flag_velocity_transfer'] == 1) |
    (df_with_stats['flag_round_tripping'] == 1)
).astype(int)

total_predicted = df_with_stats['final_predicted_anomaly'].sum()
print(f"Engine Assembly Complete! Our models have flagged a total of {total_predicted} high-risk transactions out of 10,605.")

Engine Assembly Complete! Our models have flagged a total of 600 high-risk transactions out of 10,605.


In [10]:
# 1. Re-merge the hidden answer key back to our finalized dataset using transaction_id
final_evaluation = df_with_stats.merge(ground_truth, on='transaction_id', how='left')

# 2. Calculate True Positives, False Positives, and False Negatives
tp = ((final_evaluation['final_predicted_anomaly'] == 1) & (final_evaluation['is_flagged_anomaly'] == 1)).sum()
fp = ((final_evaluation['final_predicted_anomaly'] == 1) & (final_evaluation['is_flagged_anomaly'] == 0)).sum()
fn = ((final_evaluation['final_predicted_anomaly'] == 0) & (final_evaluation['is_flagged_anomaly'] == 1)).sum()

# 3. Generate industry metrics
precision = (tp / (tp + fp)) * 100 if (tp + fp) > 0 else 0
recall = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0

print("==================================================")
print("             METRIC EVALUATION REPORT             ")
print("==================================================")
print(f"True Fraud Instances Successfully Captured (TP): {tp}")
print(f"False Alerts Raised / Normal Activity (FP):     {fp}")
print(f"Missed Fraud Instances (FN):                    {fn}")
print("-" * 50)
print(f"FINAL ENGINE RECALL:    {recall:.2f}%")
print(f"FINAL ENGINE PRECISION: {precision:.2f}%")
print("==================================================")

# Export our final flagged data as a clean file ready to plug into Power BI
final_evaluation.to_csv("../data/final_flagged_transactions.csv", index=False)
print("\nExported 'final_flagged_transactions.csv' for Dashboard Visualization!")

             METRIC EVALUATION REPORT             
True Fraud Instances Successfully Captured (TP): 116
False Alerts Raised / Normal Activity (FP):     484
Missed Fraud Instances (FN):                    94
--------------------------------------------------
FINAL ENGINE RECALL:    55.24%
FINAL ENGINE PRECISION: 19.33%

Exported 'final_flagged_transactions.csv' for Dashboard Visualization!


In [11]:
print("==================================================")
print("             RISK CONCENTRATION REPORT            ")
print("==================================================")

print("\n[1] TRUE POSITIVES BY DETECTION RULE:")
for rule in ['flag_statistical_outlier', 'flag_off_hours_withdrawal', 'flag_velocity_transfer', 'flag_round_tripping']:
    tp_rule = ((final_evaluation[rule] == 1) & (final_evaluation['is_flagged_anomaly'] == 1)).sum()
    fp_rule = ((final_evaluation[rule] == 1) & (final_evaluation['is_flagged_anomaly'] == 0)).sum()
    print(f" - {rule.replace('flag_', '').title()}: True Positives = {tp_rule}, False Positives = {fp_rule}")

print("\n[2] CONFIRMED ANOMALIES BY CHANNEL:")
channel_summary = final_evaluation[final_evaluation['is_flagged_anomaly'] == 1]['channel'].value_counts()
for channel, count in channel_summary.items():
    print(f" - {channel}: {count} cases")

print("\n[3] TOP 5 HIGH-RISK BRANCHES:")
branch_summary = final_evaluation[final_evaluation['is_flagged_anomaly'] == 1]['branch'].value_counts().head(5)
for branch, count in branch_summary.items():
    print(f" - {branch}: {count} cases")
print("==================================================")

             RISK CONCENTRATION REPORT            

[1] TRUE POSITIVES BY DETECTION RULE:
 - Statistical_Outlier: True Positives = 46, False Positives = 2
 - Off_Hours_Withdrawal: True Positives = 40, False Positives = 482
 - Velocity_Transfer: True Positives = 44, False Positives = 0
 - Round_Tripping: True Positives = 0, False Positives = 0

[2] CONFIRMED ANOMALIES BY CHANNEL:
 - Mobile App: 61 cases
 - ATM: 49 cases
 - Branch: 45 cases
 - USSD: 33 cases
 - Internet Banking: 22 cases

[3] TOP 5 HIGH-RISK BRANCHES:
 - Tudun Wada: 44 cases
 - Bauchi Rd: 40 cases
 - Pantami: 30 cases
 - Gombe Main: 26 cases
 - Lagos Island: 24 cases
